# 04 - Estratégia Experimental e Modelos de Referência

Antes de testar os modelos mais fortes, quero deixar definido como vou
comparar e escolher os modelos. Neste notebook eu:

- explico como dividi os dados;
- escolho as métricas;
- listo os testes que vou fazer nos próximos notebooks;
- avalio alguns modelos simples, que servem de referência.

## 1. Preparação

In [1]:
import sys
sys.path.append("..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from src.preprocessing import (preparar, separar_treino_teste,
                               VARIAVEIS_NUMERICAS, VARIAVEIS_CATEGORICAS, ALVO)

pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [2]:
df = preparar()
treino, teste = separar_treino_teste(df)

variaveis = VARIAVEIS_NUMERICAS + VARIAVEIS_CATEGORICAS
X_treino, y_treino = treino[variaveis], treino[ALVO]

print("treino:", len(treino), "reservas | chegadas de", treino["arrival_date"].min().date(),
      "a", treino["arrival_date"].max().date(), f"| cancelamento {y_treino.mean():.3f}")
print("teste: ", len(teste), "reservas | chegadas de", teste["arrival_date"].min().date(),
      "a", teste["arrival_date"].max().date(), f"| cancelamento {teste[ALVO].mean():.3f}")

treino: 95367 reservas | chegadas de 2015-07-01 a 2017-04-22 | cancelamento 0.361
teste:  23842 reservas | chegadas de 2017-04-22 a 2017-08-31 | cancelamento 0.409


## 2. Divisão dos dados

Na prática, o hotel tem o histórico até hoje e quer saber o risco das
reservas que ainda vão chegar. Por isso a avaliação precisa respeitar a ordem
do tempo. Divido os dados em três partes:

- validação cruzada no tempo, dentro do treino (os 80% mais antigos): uso
  para comparar os algoritmos e as configurações;
- ajuste, que são os últimos 15% do treino: uso mais adiante para escolher
  o limiar e testar a calibração;
- teste, os 20% mais recentes (chegadas a partir de 22/04/2017): uso uma
  única vez, no final, para avaliar o modelo escolhido.

Não uso corte aleatório porque ele coloca reservas do futuro no treino e
deixa o resultado otimista. No próximo notebook eu meço isso.

Na validação cruzada uso o `TimeSeriesSplit`. Cada dobra treina com um
período e valida no período seguinte. É parecido com o K-Fold, só que sem embaralhar o tempo. Uso 5 dobras:
cada validação tem cerca de 16 mil reservas (entre três e cinco meses de
chegadas) e a primeira já treina com quatro meses de histórico. Todos os
modelos usam as mesmas dobras, então a comparação é justa.

O teste fica guardado até o final. Assim o resultado dele mostra como o
modelo se sairia com reservas novas.

As dobras ficam assim:

In [3]:
cv = TimeSeriesSplit(n_splits=5)

dobras = []
for i, (tr, va) in enumerate(cv.split(X_treino), start=1):
    dobras.append({"dobra": i,
                   "treina até": treino["arrival_date"].iloc[tr].max().date(),
                   "valida de": treino["arrival_date"].iloc[va].min().date(),
                   "valida até": treino["arrival_date"].iloc[va].max().date(),
                   "reservas treino": len(tr),
                   "reservas validação": len(va),
                   "cancelamento validação": y_treino.iloc[va].mean()})
pd.DataFrame(dobras).set_index("dobra")

,treina até,valida de,valida até,reservas treino,reservas validação,cancelamento validação
dobra,,,,,,
1,2015-10-25,2015-10-25,2016-03-25,15897,15894,0.294
2,2016-03-25,2016-03-25,2016-06-21,31791,15894,0.379
3,2016-06-21,2016-06-21,2016-09-28,47685,15894,0.349
4,2016-09-28,2016-09-28,2017-01-08,63579,15894,0.384
5,2017-01-08,2017-01-08,2017-04-22,79473,15894,0.357


A primeira dobra treina com poucos meses e as seguintes vão juntando mais
histórico, como aconteceria no hotel. A taxa de cancelamento muda de uma
dobra para outra, então o comportamento das reservas muda com o tempo.

## 3. Métricas

No treino, 36% das reservas foram canceladas. Isso já é suficiente para a
acurácia enganar: um modelo que diz que ninguém cancela acerta 64%. Por isso
uso várias métricas:

- PR-AUC: mede se o modelo ordena bem as reservas, olhando para os
  cancelamentos. Não depende do limiar. É a minha métrica principal;
- ROC-AUC: também mede a ordenação. Uso como complemento;
- recall: quantos cancelamentos o modelo avisa;
- precision: quantos avisos são cancelamentos de verdade;
- F1: junta recall e precision. Uso para escolher o limiar;
- Brier: mede o erro da probabilidade. É importante porque o hotel vai usar a
  própria probabilidade, por exemplo somando para prever quantos
  cancelamentos vêm;
- acurácia: só mostro para comparar, não uso para escolher.

Para escolher entre os modelos olho a PR-AUC média na validação cruzada. Se
dois ficarem muito perto, olho o Brier e fico com o mais simples.

## 4. O que vou testar

Nos próximos passos quero:

- comparar algoritmos de famílias diferentes;
- testar configurações diferentes dos modelos;
- revisar com testes as decisões da preparação dos dados e testar
  variáveis novas;
- medir quanto o corte aleatório engana;
- testar hiperparâmetros, o peso para a classe de cancelamento e a
  calibração das probabilidades;
- escolher o limiar de decisão;
- avaliar o modelo escolhido no teste, mês a mês e por segmento;
- entender o que o modelo aprendeu e onde erra;
- ver se o modelo é útil para o hotel, nos três usos que pensei no
  notebook 01.

## 5. Pré-processamento

O pré-processamento fica dentro do pipeline de cada modelo. Assim ele é
ajustado só com o treino de cada dobra e nada da validação passa para o
treino.

- nas numéricas preencho os ausentes com a mediana, que não é afetada pelos
  valores extremos do `adr`, e depois padronizo. A padronização é necessária
  para a regressão logística e para a rede neural;
- nas categóricas preencho os ausentes com o valor mais comum e aplico
  one-hot. O `handle_unknown="ignore"` evita erro se aparecer no teste uma
  categoria que não existia no treino.

In [4]:
preproc = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), VARIAVEIS_NUMERICAS),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), VARIAVEIS_CATEGORICAS),
])

colunas = preproc.fit(X_treino).get_feature_names_out()
print("variáveis de entrada:", len(variaveis), "-> colunas depois do pré-processamento:", len(colunas))

variáveis de entrada: 33 -> colunas depois do pré-processamento: 182


## 6. Modelos de referência

Antes dos modelos mais fortes, vejo o que dá para conseguir sem modelo ou com
modelos simples:

- um modelo que não aprende (`DummyClassifier`), que sempre responde a classe
  mais comum;
- uma regra simples, tirada da EDA: marco a reserva se o depósito é não
  reembolsável, se o cliente já cancelou antes ou se a antecedência passa de
  180 dias. É o tipo de regra que alguém do hotel faria sem modelo;
- a regressão logística;
- a árvore de decisão, sem nenhum ajuste.

A EDA usou a base toda, então primeiro confiro, só com o treino, se os grupos
da regra cancelam mesmo mais.

In [5]:
def regra_simples(d):
    return ((d["deposit_type"] == "Non Refund")
            | (d["previous_cancellations"] > 0)
            | (d["lead_time"] > 180)).astype(int)

grupos = {
    "depósito não reembolsável": treino["deposit_type"] == "Non Refund",
    "já cancelou antes": treino["previous_cancellations"] > 0,
    "antecedência > 180 dias": treino["lead_time"] > 180,
    "nenhum dos três": regra_simples(treino) == 0,
}
pd.DataFrame({"% das reservas": {g: m.mean() * 100 for g, m in grupos.items()},
              "taxa de cancelamento": {g: y_treino[m].mean() for g, m in grupos.items()}})

,% das reservas,taxa de cancelamento
depósito não reembolsável,12.892,0.993
já cancelou antes,6.644,0.933
antecedência > 180 dias,18.710,0.601
nenhum dos três,73.828,0.243


Os três grupos cancelam bem mais que a média do treino (0,361), e o resto das
reservas cancela 0,243. Então a regra faz sentido.

Agora avalio as referências nas mesmas dobras. A regra não aprende nada,
então só aplico ela em cada dobra de validação. Ela também não dá
probabilidade, só sim ou não, por isso não tem PR-AUC, ROC-AUC nem Brier.

In [6]:
metricas = ["accuracy", "recall", "precision", "f1", "roc_auc", "average_precision", "neg_brier_score"]

referencias = {
    "modelo que não aprende": DummyClassifier(strategy="most_frequent"),
    "regressão logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "árvore de decisão": DecisionTreeClassifier(class_weight="balanced", random_state=42),
}

linhas = []
for nome, modelo in referencias.items():
    pipe = Pipeline([("prep", preproc), ("modelo", modelo)])
    r = cross_validate(pipe, X_treino, y_treino, cv=cv, scoring=metricas)
    linhas.append({"modelo": nome,
                   "accuracy": r["test_accuracy"].mean(),
                   "recall": r["test_recall"].mean(),
                   "precision": r["test_precision"].mean(),
                   "f1": r["test_f1"].mean(),
                   "roc_auc": r["test_roc_auc"].mean(),
                   "pr_auc": r["test_average_precision"].mean(),
                   "brier": -r["test_neg_brier_score"].mean()})

linhas_regra = []
for tr, va in cv.split(X_treino):
    y_va, pred = y_treino.iloc[va], regra_simples(treino.iloc[va])
    linhas_regra.append({"accuracy": accuracy_score(y_va, pred), "recall": recall_score(y_va, pred),
                         "precision": precision_score(y_va, pred), "f1": f1_score(y_va, pred)})
linhas.append({"modelo": "regra simples", **pd.DataFrame(linhas_regra).mean()})

pd.DataFrame(linhas).set_index("modelo")

,accuracy,recall,precision,f1,roc_auc,pr_auc,brier
modelo,,,,,,,
modelo que não aprende,0.647,0.000,0.000,0.000,0.500,0.353,0.353
regressão logística,0.804,0.622,0.783,0.689,0.872,0.819,0.137
árvore de decisão,0.774,0.549,0.744,0.626,0.725,0.572,0.225
regra simples,0.727,0.458,0.697,0.548,NaN,NaN,NaN


- o modelo que não aprende tem acurácia de 0,647, mas recall 0, porque não
  avisa nenhum cancelamento. Isso mostra por que não dá para olhar só a
  acurácia;
- a regra simples já ajuda: pega 46% dos cancelamentos e 70% dos avisos dela
  estão certos;
- a regressão logística é melhor que a regra em tudo (recall 0,622, precision
  0,783 e F1 0,689) e ainda dá uma probabilidade (PR-AUC 0,819 e Brier
  0,137);
- a árvore sem ajuste fica atrás da regressão logística e tem o pior Brier.
  Ela decora o treino e quase sempre dá probabilidade 0 ou 1. No próximo notebook testo se limitar a árvore resolve isso.

## 7. Conclusões

- a divisão dos dados, as métricas e os testes estão definidos;
- o `preproc` e as dobras `cv` são os mesmos daqui para frente;
- o modelo a superar é a regressão logística, com PR-AUC 0,819;
- a regra simples também vai servir de referência quando eu for ver se o
  modelo é útil para o hotel.